# 🤖 Notebook 7: 5-Agent LLM Validation (Gemini API)

**Goal:** Use 5 focused AI agents to review and validate every pricing parameter stored in the registry.

Each agent reads the `pricing_registry.json` plus the relevant results JSON and provides:
- ✅ What looks correct
- ⚠️ What to watch out for
- 🔧 Recommended adjustments (if any)

**Inputs:** `outputs/pricing_registry.json` + all intermediate results JSONs

---

## Setup: Your Gemini API Key

In [9]:
!pip install google-generativeai

In [10]:
import json, os

# ─── SECURE GEMINI API KEY INPUT ──────────────────────────────────────────
# Check environment variable first, or prompt with an input box
GEMINI_API_KEY = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY', '')
if not GEMINI_API_KEY or 'YOUR_' in GEMINI_API_KEY or 'ENTER' in GEMINI_API_KEY:
    GEMINI_API_KEY = input('🔑 Please enter your Gemini API Key (or press Enter to skip): ').strip()

GEMINI_MODEL = 'gemini-1.5-flash'  # or 'gemini-2.0-flash'
# ──────────────────────────────────────────────────────────────────────────

if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    llm = genai.GenerativeModel(GEMINI_MODEL)
    print('✅ Gemini API authenticated successfully.')
else:
    llm = None
    print('⚠️ Running in simulated mode (no API key provided).')

# Load all results
registry  = json.load(open('../outputs/pricing_registry.json'))
freq_res  = json.load(open('../outputs/frequency_results.json'))
sev_res   = json.load(open('../outputs/severity_results.json'))
cred_res  = json.load(open('../outputs/credibility_results.json'))

print('✅ Loaded all pricing results.')
print(json.dumps(registry, indent=2))


---
## Agent 1 — Segmentation Agent

In [11]:
import pandas as pd

def tool_format_agent_prompt(role_title, context_header, context_payload, review_questions):
    """
    Formats structured audit prompt dossiers for AI actuarial review agents.
    
    What this tool does:
        Assembles domain-specific metadata and questions into standardized prompt template
        tailored for LLM peer-review under ASOP 41 communications standards.
        
    Returns:
        str: Assembled agent prompt.
    """
    questions_str = '\n'.join(f'{i+1}. {q}' for i, q in enumerate(review_questions))
    prompt = (
        f"You are an expert {role_title}.\n"
        f"{context_header}\n\n"
        f"{context_payload}\n\n"
        f"Please review:\n{questions_str}\n\n"
        f"Be concise and actuarially precise."
    )
    return prompt

# ── Run Execution ─────────────────────────────────────────────────────────────
segment_profiles = pd.read_csv('../outputs/segment_profiles.csv').to_dict(orient='records')

prompt_seg = tool_format_agent_prompt(
    role_title='Actuarial Risk Segmentation specialist',
    context_header=f"You have run a K-Prototypes clustering on a motor insurance portfolio.\nK: {registry.get('k_clusters', 5)} | Total training policies: {registry['n_train']:,}",
    context_payload=f"Cluster profiles:\n{json.dumps(segment_profiles, indent=2)}",
    review_questions=[
        'Is the chosen K reasonable for this portfolio size?',
        'Are the risk clusters meaningful (clear differentiation in claim frequency)?',
        'Are any segments dangerously small (potential credibility issues)?',
        'What do you recommend?'
    ]
)

if llm is not None:
    response_seg = llm.generate_content(prompt_seg)
    seg_output = response_seg.text
else:
    seg_output = '✅ [SIMULATED AUDIT] K-Prototypes clustering verified. Segments show clear frequency risk differentiation with acceptable sample sizes.'
print('=== 🔷 AGENT 1: SEGMENTATION VALIDATION ===')
print(seg_output)


---
## Agent 2 — Frequency Agent

In [3]:
prompt_freq = f"""
You are an expert Actuarial Frequency Modelling specialist.
You have compared three frequency models on a motor insurance portfolio.

Portfolio zero-claim rate: {registry.get('zero_claim_pct', 'N/A')}
Total training policies: {registry['n_train']:,}

Model comparison results:
{json.dumps(freq_res['all'], indent=2)}

Best model chosen: {registry['best_freq_model']}

Please review:
1. Was the best model choice appropriate? Why or why not?
2. Is the Gini coefficient of {registry['freq_gini']} acceptable for this type of data?
3. Should we be concerned about overdispersion given the zero-claim rate?
4. What do you recommend for improving the frequency model?

Be concise and actuarially precise.
"""

if llm is not None:
    response_freq = llm.generate_content(prompt_freq)
    freq_output = response_freq.text
else:
    freq_output = f"✅ [SIMULATED AUDIT] Frequency model ({registry.get('best_freq_model', 'GLM')}) selected appropriately. Overdispersion is handled."
print('=== 📈 AGENT 2: FREQUENCY MODEL VALIDATION ===')
print(freq_output)

---
## Agent 3 — Severity Agent

In [4]:
prompt_sev = f"""
You are an expert Actuarial Severity Modelling specialist.
You have fitted a Gamma GLM with log link (ClaimNb-weighted) to model average claim severity.

Gamma GLM Results:
{json.dumps(sev_res, indent=2)}

Mean predicted severity: {sev_res['mean_pred_severity']}
Training on: {sev_res['n_train']:,} positive-claim records only

Please review:
1. Is the Gamma distribution appropriate for this severity pattern?
2. Does the AIC/Deviance suggest a good fit?
3. Should we be concerned about large outlier claims (fat tail risk)?
4. Is the mean predicted severity reasonable for motor insurance?
5. Recommendations?

Be concise and actuarially precise.
"""

if llm is not None:
    response_sev = llm.generate_content(prompt_sev)
    sev_output = response_sev.text
else:
    sev_output = f"✅ [SIMULATED AUDIT] Severity model ({registry.get('best_sev_model', 'GLM')}) fitted with acceptable Gamma deviance."
print('=== 🔴 AGENT 3: SEVERITY MODEL VALIDATION ===')
print(sev_output)

---
## Agent 4 — Credibility Agent

In [5]:
# Show only the band summary (not full dict) to keep prompt concise
ae_summary = [{'Band': r['Band'], 'n': r['n'], 'Raw_AE': r['Raw_AE'],
               'Z_credibility': r['Z_credibility'], 'Smooth_Factor': r['Smooth_Factor']}
              for r in cred_res['ae_table'] if r['n'] > 0]

prompt_cred = f"""
You are an expert Actuarial Credibility specialist.
You have built a Bühlmann credibility-weighted risk adjustment using Isolation Forest outlier scores.

Parameters used:
- Credibility constant K = {cred_res['K']} (Bühlmann parameter)
- Number of A/E bands = {cred_res['N_BINS']}
- Normalisation constant = {cred_res['norm_const']} (should be ~1.0)
- Overall portfolio A/E ratio = {cred_res['overall_ae']}

A/E table by outlier band:
{json.dumps(ae_summary, indent=2)}

Please review:
1. Is the overall A/E ratio of {cred_res['overall_ae']} acceptable?
2. Is K = {cred_res['K']} an appropriate credibility constant for these band sizes?
3. Is the monotone smoothing applied correctly? Are the smooth factors plausible?
4. Does the revenue-neutral normalisation ({cred_res['norm_const']}) indicate the adjustment is unbiased?
5. Recommendations?

Be concise and actuarially precise.
"""

if llm is not None:
    response_cred = llm.generate_content(prompt_cred)
    cred_output = response_cred.text
else:
    cred_output = '✅ [SIMULATED AUDIT] Buhlmann credibility factors smoothly transition thin bands. Revenue-neutrality verified.'
print('=== ⚖️ AGENT 4: CREDIBILITY CALIBRATION VALIDATION ===')
print(cred_output)

---
## Agent 5 — Final Parameters & Auditor Agent

In [6]:
prompt_final = f"""
You are the Chief Pricing Actuary and final auditor of a motor insurance pricing engine.
You are reviewing the complete Pricing Parameter Registry before sign-off.

Complete Pricing Registry:
{json.dumps(registry, indent=2)}

Final Premium Formula:
Final Premium = Pure Premium × {registry['large_loss_loading']} (large loss loading) × Risk Adj Factor × {registry['profit_margin']} (profit)

Mean training portfolio premium: {registry['mean_train_premium']}
Mean submission file premium: {registry['mean_submit_premium']}
Premium floor: {registry['premium_floor']}, Premium cap: {registry['premium_cap']}

Please audit:
1. Is the large loss loading of {registry['large_loss_loading']} reasonable? Should it be higher or lower?
2. Is the profit margin of {registry['profit_margin']} (i.e. {(registry['profit_margin']-1)*100:.0f}%) appropriate?
3. Is the premium floor of {registry['premium_floor']} reasonable?
4. Is the premium cap of {registry['premium_cap']} appropriate to prevent adverse selection?
5. Is there a meaningful difference between the training mean premium ({registry['mean_train_premium']}) and submission mean premium ({registry['mean_submit_premium']})? Is this a concern?
6. Overall sign-off: Are these parameters suitable for a commercial motor insurance pricing submission?
7. Final recommendations.

Be thorough but concise. Provide a clear GO / CONDITIONAL GO / NO-GO recommendation.
"""

if llm is not None:
    response_final = llm.generate_content(prompt_final)
    final_output = response_final.text
else:
    final_output = '✅ [SIMULATED AUDIT] CONDITIONAL GO: Large loss loading (1.10) and profit margin (1.05) are in line with commercial standards. Ready for actuary sign-off.'
print('=== 💰 AGENT 5: FINAL PARAMETERS & AUDIT ===')
print(final_output)

---
## Save Agent Validation Report

In [7]:
validation_report = {
    'Segmentation Agent':           seg_output,
    'Frequency Agent':              freq_output,
    'Severity Agent':               sev_output,
    'Credibility Agent':            cred_output,
    'Final Parameters Audit Agent': final_output,
}

with open('../outputs/agent_validation_report.json', 'w') as f:
    json.dump(validation_report, f, indent=2)

for agent, response in validation_report.items():
    print(f'\n{"="*60}')
    print(f'  {agent}')
    print(f'{"="*60}')
    print(response[:500], '...' if len(response) > 500 else '')

print('\n✅ Saved outputs/agent_validation_report.json')


---
## 🔒 HUMAN-IN-THE-LOOP: Actuarial Sign-Off Gate

As per Tier 2 AI Governance standards, the AI agents advise, but the Appointed Actuary decides. 
The AI model is blocked from writing directly to the billing database. You must review the `agent_validation_report.json` and formally sign off.

In [ ]:
import datetime

print("=========================================================")
print(" 🚨 HUMAN REVIEW REQUIRED: FINAL PRICING SIGN-OFF 🚨 ")
print("=========================================================")
print("\nPlease review the 5-Agent AI audit report generated above.")
print("As the signing actuary, you bear final accountability.")

# Interactive prompt for the human actuary
human_decision = input("Do you APPROVE these pricing parameters for production? (Type 'APPROVE', 'REJECT', or 'CONDITIONAL'): ").strip().upper()

human_notes = ""
if human_decision != 'APPROVE':
    human_notes = input("Please enter the reason for rejection or the required conditions: ")

audit_trail = {
    "timestamp": datetime.datetime.now().isoformat(),
    "decision": human_decision,
    "reviewer_notes": human_notes,
    "base_registry_snapshot": registry,
    "ai_validation_report": validation_report
}

if human_decision == 'APPROVE':
    with open('../outputs/APPROVED_production_registry.json', 'w') as f:
        json.dump(audit_trail, f, indent=2)
    print("\n✅ SIGN-OFF COMPLETE: The pricing registry has been approved and securely locked for the production billing system.")
elif human_decision == 'CONDITIONAL':
    with open('../outputs/CONDITIONAL_production_registry.json', 'w') as f:
        json.dump(audit_trail, f, indent=2)
    print("\n⚠️ CONDITIONAL APPROVAL: The registry has been queued for manual overrides. See notes:", human_notes)
else:
    print("\n❌ REJECTED: The pricing update has been halted. Do not deploy.")
